# Chapter 14 — Telemetry and Tracing: The Full Decision Trail

**Monograph:** *How to Build Policy-Governed AI Agents*

**Prerequisites:**
- Chapter 2 notebook completed (GMS knowledge store).
- Chapter 7 notebook completed (GovernedToolGateway).
- Chapter 8 notebook completed (ClaimVerifier).
- Chapter 12 concepts (PolicyEngine) and Chapter 13 concepts (behavioral contracts).

**What you will build:**
- `governed_agent/tracing.py` — the `GovernanceTracer` class with `AuditPacket` provenance records.

**Key idea:** Every governance decision — every tool call allowed or blocked, every claim verified or rejected, every state transition — is recorded as a queryable audit trail. The tracer wraps DOETracingGMS so that every event also becomes a geometric triple in a dedicated tracing store.

## Preamble

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

## Build the Knowledge Store

We reuse the same knowledge document from Chapter 2. The store provides the geometric authority for policy checks and verification.

In [ ]:
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.knowledge.ingest import ingest_document
from knowlytix.core.config import GeometryConfig, TrainConfig

MD_PATH = "governed_agent/knowledge/data_science_practices.md"

config = DocGMSConfig(
    store_path="governance_monograph/lab_stores/ch14_telemetry",
    geometry=GeometryConfig(m=128),
    train=TrainConfig(epochs=200, device=str(DEVICE)),
)

store = GMSExpertStore(config, device=DEVICE)
if not store.load():
    print("Building store from scratch (200 epochs on GPU)...")
    result = ingest_document(store, MD_PATH, llm=None, config=config, device=DEVICE)
    store.save()

stats = store.stats()
print("\n=== Knowledge Store Statistics ===")
print(f"  Entities:  {stats['entities']}")
print(f"  Relations: {stats['relations']}")
print(f"  Triples:   {stats['triples']}")
print(f"  ENM:       {stats['enm_entries']}")

## Train Models and Record Metrics

We train three classifiers on the **Wine** dataset (178 samples, 13 features, 3 classes)
and record their metrics in the GMS store. The Wine dataset produces differentiated
accuracy scores, making verification claims and model comparisons pedagogically meaningful.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from knowlytix.knowledge.learn import RuntimeLearner
from knowlytix.knowledge.extract import ExtractedTriple

wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, test_size=0.4, random_state=42)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=30, max_depth=3, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=42),
}

metrics = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="weighted")
    metrics[name] = {"accuracy": round(acc, 6), "f1": round(f1, 6)}
    print(f"  {name:25s}  accuracy={acc:.4f}  f1={f1:.4f}")

# Record in GMS so claims can be verified against the store
learner = RuntimeLearner(store, config.learn)
training_triples = []
for model_name, model_metrics in metrics.items():
    for metric_name, value in model_metrics.items():
        training_triples.append(ExtractedTriple(
            head=model_name, relation=f"has_{metric_name}",
            tail=f"{value}", numeric_value=value))
    training_triples.append(ExtractedTriple(
        head=model_name, relation="trained_on", tail="wine_dataset"))

result = learner.learn_triples(training_triples, device=DEVICE)
print(f"\nRuntimeLearner accepted: {result.accepted}")
print(f"  Triples added:  {result.n_triples_added}")
store.save()


## Set Up the Governed Pipeline with Tracing

We create:
1. A **GovernedToolGateway** with policy rules (including one that will block a tool).
2. A **GovernanceTracer** to record every event.
3. A **ClaimVerifier** to verify claims after tool execution.

In [ ]:
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.governed_gateway import GovernedToolGateway
from knowlytix.harness.governance.verifier import ClaimVerifier, ClaimType, Claim
from knowlytix.harness.governance.tracing import GovernanceTracer
from knowlytix.harness.testing.policy import PolicyEngine, PolicyRule

# Create session and policy engine
session = SessionState()
policy = PolicyEngine(store=store)

# Encode policy rules
rules = [
    PolicyRule(
        rule_id="block_predict_pending_approval",
        action="predict",
        requirement="prediction_approval",
        rule_type="blocks",
        severity="error",
    ),
    PolicyRule(
        rule_id="block_fairness_without_protected",
        action="diagnose_fairness",
        requirement="protected_attribute_loaded",
        rule_type="blocks",
        severity="error",
    ),
]
policy.encode_rules(rules)

# Create the governed gateway
gateway = GovernedToolGateway(
    session=session,
    store=store,
    policy=policy,
    theta_plausibility=1.5,
    device=str(DEVICE),
)

# Create tracer and verifier
tracer = GovernanceTracer(tracing_store=None)  # in-memory only for this lab
verifier = ClaimVerifier(store, device=str(DEVICE))

print(f"GovernedToolGateway: theta={gateway.theta_plausibility}")
print(f"PolicyEngine: {len(rules)} rules encoded")
print("GovernanceTracer: initialized")
print(f"ClaimVerifier: {len(verifier._dispatch)} claim type handlers")


---
## Lab 14.1 --- Generate and Inspect the Full Audit Trail

We run a complete governed pipeline through the `GovernedToolGateway`, recording
every event via the `GovernanceTracer`. The pipeline uses the **Wine** dataset
(loaded and preprocessed via the governed tool gateway) and one step is
designed to trigger a policy block:

- `predict` is blocked by the `block_predict_pending_approval` rule, simulating
  an organizational requirement that predictions require explicit approval.

The remaining six steps --- load, preprocess, train, diagnose robustness,
explain feature importance and data summary --- pass all four gates.

In [5]:
RUN_ID = "ch14_demo_run"
tracer.start_run(RUN_ID, description="Chapter 14 governed pipeline demo")

# Define the pipeline: tool_name, args, context, expected state transition
pipeline_steps = [
    ("load_dataset",       {"name": "wine", "builtin": "wine", "test_size": 0.4, "random_state": 42},
     "data_loading",       "init",          "data_loaded"),
    ("preprocess_data",    {"dataset": "wine"},
     "preprocessing",      "data_loaded",   "preprocessed"),
    ("train_model",        {"dataset": "wine", "model_type": "RandomForestClassifier",
                            "model_name": "RandomForest",
                            "params": {"n_estimators": 30, "max_depth": 3, "random_state": 42}},
     "model_training",     "preprocessed",  "model_trained"),
    # Intentional policy violation: predict requires approval
    ("predict",            {"dataset": "wine", "model_name": "RandomForest"},
     "prediction",         "model_trained", "model_trained"),
    ("diagnose_robustness",{"dataset": "wine", "model_name": "RandomForest"},
     "diagnostics",        "model_trained", "diagnosed"),
    ("explain_importance", {"dataset": "wine", "model_name": "RandomForest"},
     "explanation",        "diagnosed",     "explained"),
    ("data_summary",       {"dataset": "wine"},
     "review",             "explained",     "reviewed"),
]

print(f"Pipeline: {len(pipeline_steps)} steps planned")
print(f"Run ID:   {RUN_ID}")
print("Expected: 1 blocked (predict), 6 allowed\n")
print(f"{'Step':<5s} {'Tool':<22s} {'Context':<18s} {'State Transition'}")
print("-" * 72)
for i, (tool, args, ctx, s_before, s_after) in enumerate(pipeline_steps, 1):
    print(f"{i:<5d} {tool:<22s} {ctx:<18s} {s_before} -> {s_after}")


Pipeline: 7 steps planned
Run ID:   ch14_demo_run
Expected: 1 blocked (predict), 6 allowed

Step  Tool                   Context            State Transition
------------------------------------------------------------------------
1     load_dataset           data_loading       init -> data_loaded
2     preprocess_data        preprocessing      data_loaded -> preprocessed
3     train_model            model_training     preprocessed -> model_trained
4     predict                prediction         model_trained -> model_trained
5     diagnose_robustness    diagnostics        model_trained -> diagnosed
6     explain_importance     explanation        diagnosed -> explained
7     data_summary           review             explained -> reviewed


### Execute the Pipeline with Full Tracing

In [6]:

current_state = "init"

print(f"{'Step':<5s} {'Tool':<22s} {'Decision':<10s} {'Time(ms)':<10s} {'Reason'}")
print("=" * 85)

for step_num, (tool_name, args, context, state_before, state_after) in enumerate(
    pipeline_steps, 1
):
    # Record state before
    tracer.trace_state_transition(RUN_ID, current_state, state_before)

    # Execute through the governed gateway
    gw_result = gateway.execute(tool_name, args, context)

    # Determine policy decision and blocking rules
    if gw_result.blocked:
        decision = "blocked"
        blocking = []
        if gw_result.policy_decision is not None and not gw_result.policy_decision.allowed:
            blocking = gw_result.policy_decision.blocking_rules
        result_summary = gw_result.reason
        next_state = current_state  # blocked: state unchanged
    else:
        decision = "allowed"
        blocking = []
        result_summary = gw_result.display[:100] if gw_result.display else "OK"
        next_state = state_after

    # Collect GMS scores
    gms_scores = {}
    if gw_result.plausibility_score is not None:
        gms_scores["plausibility"] = gw_result.plausibility_score

    # Record the audit packet
    packet = tracer.trace_tool_call(
        run_id=RUN_ID,
        step_num=step_num,
        tool_name=tool_name,
        args=args,
        result_summary=result_summary,
        policy_decision=decision,
        blocking_rules=blocking,
        gms_scores=gms_scores,
        gate_results=gw_result.gate_results,
        execution_time_ms=gw_result.execution_time_ms,
        state_before=current_state,
        state_after=next_state,
    )

    # Record state transition
    tracer.trace_state_transition(RUN_ID, current_state, next_state)
    current_state = next_state

    # Print step result
    reason_short = gw_result.reason[:40]
    print(f"{step_num:<5d} {tool_name:<22s} {decision:<10s} "
          f"{gw_result.execution_time_ms:<10.1f} {reason_short}")

tracer.end_run(RUN_ID)
print(f"\nPipeline complete. Final state: {current_state}")

Step  Tool                   Decision   Time(ms)   Reason
1     load_dataset           allowed    2.3        All gates passed
2     preprocess_data        allowed    2.5        All gates passed
3     train_model            allowed    16.2       All gates passed
4     predict                blocked    0.1        Policy violation: Rule block_predict_pen
5     diagnose_robustness    allowed    3.8        All gates passed


6     explain_importance     allowed    1463.9     All gates passed
7     data_summary           allowed    19.8       All gates passed

Pipeline complete. Final state: reviewed


### Add Verification Results to the Audit Trail

For allowed tool calls, we build claims and verify them against the GMS store. The verification results are attached to the corresponding audit packets.

In [7]:
# Define claims for steps that produced results
step_claims = {
    1: [  # load_dataset
        Claim(type=ClaimType.FACT,
              text="Wine dataset loaded for classification",
              metadata={"head": "wine_dataset", "relation": "trained_on",
                        "tail": "RandomForest"}),
    ],
    3: [  # train_model
        Claim(type=ClaimType.FACT,
              text="RandomForest was trained on wine dataset",
              metadata={"head": "RandomForest", "relation": "trained_on",
                        "tail": "wine_dataset"}),
        Claim(type=ClaimType.NUMERIC,
              text="RandomForest test accuracy is 0.9861",
              metadata={"category": "has_accuracy",
                        "entity_id": "RandomForest/0.986111",
                        "claimed_value": 0.9861}),
        Claim(type=ClaimType.THRESHOLD,
              text="RandomForest accuracy meets 0.90 minimum",
              metadata={"category": "has_accuracy",
                        "entity_id": "RandomForest/0.986111",
                        "operator": "ge", "limit": 0.90}),
    ],
    5: [  # diagnose_robustness
        Claim(type=ClaimType.NUMERIC,
              text="GradientBoosting accuracy is 0.9722",
              metadata={"category": "has_accuracy",
                        "entity_id": "GradientBoosting/0.972222",
                        "claimed_value": 0.9722}),
        Claim(type=ClaimType.COMPARISON,
              text="RandomForest outperforms GradientBoosting on accuracy",
              metadata={"category": "has_accuracy",
                        "entity_a": "RandomForest/0.986111",
                        "entity_b": "GradientBoosting/0.972222",
                        "metric": "has_accuracy",
                        "direction": "greater"}),
    ],
}

print(f"{'Step':<5s} {'Claims':<8s} {'Passed':<8s} {'Failed':<8s} {'Pass Rate'}")
print("-" * 45)

for step_num, claims in step_claims.items():
    overall = verifier.verify_all(claims)

    # Build verdict dicts for the tracer
    verdict_dicts = [
        {"claim_text": v.claim.text,
         "passed": v.passed,
         "score": v.score,
         "severity": v.severity,
         "evidence": v.evidence}
        for v in overall.claim_verdicts
    ]
    claim_dicts = [{"text": c.text, "type": c.type.value} for c in claims]

    tracer.trace_verification(
        run_id=RUN_ID,
        step_num=step_num,
        claims=claim_dicts,
        verdicts=verdict_dicts,
    )

    print(f"{step_num:<5d} {overall.n_total:<8d} {overall.n_passed:<8d} "
          f"{overall.n_total - overall.n_passed:<8d} {overall.pass_rate:.1%}")


Step  Claims   Passed   Failed   Pass Rate
---------------------------------------------
1     1        0        1        0.0%
3     3        2        1        66.7%
5     2        2        0        100.0%


### Extract the Full Audit Trail

The `get_audit_trail()` method returns a chronological list of `AuditPacket` objects, each containing the complete provenance chain for a single tool call.

In [8]:
trail = tracer.get_audit_trail(RUN_ID)
print(f"Audit trail: {len(trail)} packets\n")

print(f"{'Step':<5s} {'Tool':<22s} {'Decision':<10s} {'Gates':<30s} "
      f"{'Plausibility':<14s} {'Verification'}")
print("=" * 100)

for pkt in trail:
    # Format gate results
    gates = "  ".join(
        f"{g}={'PASS' if v else 'FAIL'}" for g, v in pkt.gate_results.items()
    )
    
    # Format plausibility
    plaus = pkt.gms_scores.get("plausibility")
    plaus_str = f"{plaus:.4f}" if plaus is not None else "N/A"
    
    # Format verification
    if pkt.verification is not None:
        v = pkt.verification
        verif_str = f"{v['n_passed']}/{v['n_claims']} ({v['pass_rate']:.0%})"
    else:
        verif_str = "-"
    
    print(f"{pkt.step_num:<5d} {pkt.tool_name:<22s} {pkt.policy_decision:<10s} "
          f"{gates:<30s} {plaus_str:<14s} {verif_str}")

Audit trail: 7 packets

Step  Tool                   Decision   Gates                          Plausibility   Verification
1     load_dataset           allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            0/1 (0%)
2     preprocess_data        allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            -
3     train_model            allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            2/3 (67%)
4     predict                blocked    schema=PASS  policy=FAIL       N/A            -
5     diagnose_robustness    allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            2/2 (100%)
6     explain_importance     allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            -
7     data_summary           allowed    schema=PASS  policy=PASS  plausibility=PASS  fault_injection=PASS N/A            -


### Print the Formatted Audit Report

In [9]:
report = tracer.summary_report(RUN_ID)
print(report)

  GOVERNANCE AUDIT REPORT  --  Run: ch14_demo_run

  Total tool calls:       7
  Allowed:                6
  Blocked:                1
  Block rate:             14.3%

  Total claims verified:  6
  Claims passed:          4
  Avg verification score: 0.583

  State transitions:      14
  Total execution time:   1508.7 ms
  Run elapsed time:       1.51 s

  Lowest-confidence step: 1 (pass_rate=0.000)

  --- Policy Violations ---
  Step 4: predict blocked by [block_predict_pending_approval, gms_graph:predict->blocks->prediction_approval]


### State Transitions

In [10]:
transitions = tracer.get_transitions(RUN_ID)
print(f"State transitions: {len(transitions)}\n")

print(f"{'#':<4s} {'From':<18s} {'To':<18s}")
print("-" * 40)
for i, t in enumerate(transitions, 1):
    print(f"{i:<4d} {t['from_state']:<18s} {t['to_state']:<18s}")

State transitions: 14

#    From               To                
----------------------------------------
1    init               init              
2    init               data_loaded       
3    data_loaded        data_loaded       
4    data_loaded        preprocessed      
5    preprocessed       preprocessed      
6    preprocessed       model_trained     
7    model_trained      model_trained     
8    model_trained      model_trained     
9    model_trained      model_trained     
10   model_trained      diagnosed         
11   diagnosed          diagnosed         
12   diagnosed          explained         
13   explained          explained         
14   explained          reviewed          


---
## Lab 14.2 — Query the Trace for Governance Questions

An audit trail is only useful if it can answer governance questions. We demonstrate eight queries that a model risk manager or auditor might ask, all answered from the trace data.

### Question 1: How many tool calls were made?

In [11]:
trail = tracer.get_audit_trail(RUN_ID)
print(f"Q1: Total tool calls = {len(trail)}")

Q1: Total tool calls = 7


### Question 2: How many were policy-blocked?

In [12]:
blocked = tracer.query_blocked_calls(RUN_ID)
allowed = tracer.query_allowed_calls(RUN_ID)
print(f"Q2: Policy-blocked calls = {len(blocked)}")
print(f"    Allowed calls         = {len(allowed)}")
print(f"    Block rate            = {len(blocked)/len(trail):.1%}")
print()
for b in blocked:
    rules = ", ".join(b.blocking_rules) if b.blocking_rules else "gate failure"
    print(f"    Step {b.step_num}: {b.tool_name} — blocked by [{rules}]")

Q2: Policy-blocked calls = 1
    Allowed calls         = 6
    Block rate            = 14.3%

    Step 4: predict — blocked by [block_predict_pending_approval, gms_graph:predict->blocks->prediction_approval]


### Question 3: What was the average verification score?

In [13]:
all_scores = []
for v_rec in tracer.get_verifications(RUN_ID):
    for verdict in v_rec.get("verdicts", []):
        score = verdict.get("score")
        if score is not None:
            all_scores.append(score)

avg_score = sum(all_scores) / len(all_scores) if all_scores else 0.0
print(f"Q3: Average verification score = {avg_score:.3f}")
print(f"    Total claims verified       = {len(all_scores)}")
print(f"    Score range                 = [{min(all_scores):.3f}, {max(all_scores):.3f}]")

Q3: Average verification score = 0.583
    Total claims verified       = 6
    Score range                 = [0.000, 1.000]


### Question 4: Which step had the lowest confidence?

In [14]:
lowest_step = None
lowest_rate = float("inf")

for pkt in trail:
    if pkt.verification is not None:
        rate = pkt.verification["pass_rate"]
        if rate < lowest_rate:
            lowest_rate = rate
            lowest_step = pkt

if lowest_step is not None:
    print(f"Q4: Lowest-confidence step = Step {lowest_step.step_num} "
          f"({lowest_step.tool_name})")
    print(f"    Pass rate              = {lowest_rate:.1%}")
    print(f"    Claims: {lowest_step.verification['n_passed']}/"
          f"{lowest_step.verification['n_claims']}")
else:
    print("Q4: No verification data available")

Q4: Lowest-confidence step = Step 1 (load_dataset)
    Pass rate              = 0.0%
    Claims: 0/1


### Question 5: Were all required tools executed?

In [15]:
required_tools = [
    "load_dataset", "preprocess_data", "train_model",
    "diagnose_accuracy", "explain_importance",
]

executed = {pkt.tool_name for pkt in allowed}
missing = [t for t in required_tools if t not in executed]

print(f"Q5: Required tools = {len(required_tools)}")
print(f"    Executed        = {len(executed & set(required_tools))}")
print(f"    Missing         = {len(missing)}")
if missing:
    print(f"    Missing tools: {missing}")
else:
    print("    All required tools were executed.")

Q5: Required tools = 5
    Executed        = 4
    Missing         = 1
    Missing tools: ['diagnose_accuracy']


### Question 6: What was the total execution time?

In [16]:
total_ms = sum(pkt.execution_time_ms for pkt in trail)
meta = tracer._run_meta.get(RUN_ID, {})
elapsed_s = meta.get("elapsed_s", 0.0)

print(f"Q6: Total tool execution time = {total_ms:.1f} ms")
print(f"    Run wall-clock time       = {elapsed_s:.2f} s")
print(f"    Overhead (tracing + GMS)   = {max(0, elapsed_s * 1000 - total_ms):.1f} ms")

Q6: Total tool execution time = 1508.7 ms
    Run wall-clock time       = 1.51 s
    Overhead (tracing + GMS)   = 6.0 ms


### Question 7: Were there any policy violations?

In [17]:
violations = tracer.query_blocked_calls(RUN_ID)

print(f"Q7: Policy violations = {len(violations)}")
if violations:
    print()
    for v in violations:
        rules = ", ".join(v.blocking_rules) if v.blocking_rules else "gate failure"
        print(f"    Step {v.step_num}: {v.tool_name}")
        print(f"      Reason: {v.result_summary[:80]}")
        print(f"      Rules:  [{rules}]")
        print(f"      Gates:  {v.gate_results}")
else:
    print("    No policy violations detected.")

Q7: Policy violations = 1

    Step 4: predict
      Reason: Policy violation: Rule block_predict_pending_approval: 'predict' blocked by 'pre
      Rules:  [block_predict_pending_approval, gms_graph:predict->blocks->prediction_approval]
      Gates:  {'schema': True, 'policy': False}


### Question 8: What was the final release decision?

In [18]:
# Aggregate all verification results
total_claims = 0
total_passed = 0
has_critical = False

for v_rec in tracer.get_verifications(RUN_ID):
    for verdict in v_rec.get("verdicts", []):
        total_claims += 1
        if verdict.get("passed", False):
            total_passed += 1
        if verdict.get("severity") == "critical":
            has_critical = True

pass_rate = total_passed / total_claims if total_claims > 0 else 0.0
n_blocked = len(tracer.query_blocked_calls(RUN_ID))

# Release decision logic
if has_critical:
    release = "BLOCKED — critical severity finding"
elif n_blocked > 0 and pass_rate < 0.80:
    release = "BLOCKED — policy violations + low pass rate"
elif pass_rate >= 0.90:
    release = "APPROVED"
elif pass_rate >= 0.80:
    release = "CONDITIONAL — review required"
else:
    release = "BLOCKED — pass rate below 80%"

print(f"Q8: Release decision = {release}")
print(f"    Verification pass rate: {pass_rate:.1%} ({total_passed}/{total_claims})")
print(f"    Policy violations:      {n_blocked}")
print(f"    Critical findings:      {has_critical}")

Q8: Release decision = BLOCKED — policy violations + low pass rate
    Verification pass rate: 66.7% (4/6)
    Policy violations:      1
    Critical findings:      False


### Governance Dashboard Table

Assemble all eight answers into a single dashboard view.

In [19]:
import pandas as pd

# Build the dashboard from audit packets
dashboard_rows = tracer.dashboard_table(RUN_ID)
df = pd.DataFrame(dashboard_rows)
print("=== Governance Dashboard ===")
print(df.to_string(index=False))

# Summary stats
print("\n=== Governance Summary ===")
summary = {
    "Total tool calls": len(trail),
    "Allowed": len(allowed),
    "Blocked": len(blocked),
    "Block rate": f"{len(blocked)/len(trail):.1%}",
    "Claims verified": total_claims,
    "Claims passed": total_passed,
    "Avg verification score": f"{avg_score:.3f}",
    "Lowest-confidence step": f"Step {lowest_step.step_num} ({lowest_step.tool_name})" if lowest_step else "N/A",
    "Required tools executed": f"{len(executed & set(required_tools))}/{len(required_tools)}",
    "Total execution time": f"{total_ms:.1f} ms",
    "Policy violations": len(violations),
    "Release decision": release,
}

for k, v in summary.items():
    print(f"  {k:<30s}: {v}")

=== Governance Dashboard ===
 Step                Tool Decision                                                                 Blocking Rules Plausibility  Gate: Schema  Gate: Policy Gate: Plausibility Verification Time (ms)                          State
    1        load_dataset  allowed                                                                                         N/A          True          True               True         0.0%       2.3            init -> data_loaded
    2     preprocess_data  allowed                                                                                         N/A          True          True               True                    2.5    data_loaded -> preprocessed
    3         train_model  allowed                                                                                         N/A          True          True               True        66.7%      16.2  preprocessed -> model_trained
    4             predict  blocked block_predict_pending_ap

---
## Exercises

### Exercise 14.1 — Timeline Visualization

Build a visual timeline showing tool calls (green = allowed, red = blocked), state transitions and verification scores.

In [20]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

trail = tracer.get_audit_trail(RUN_ID)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

# Top: tool calls colored by decision
steps = [pkt.step_num for pkt in trail]
colors = ["#2ecc71" if pkt.policy_decision == "allowed" else "#e74c3c"
          for pkt in trail]
labels = [pkt.tool_name for pkt in trail]

ax1.barh(steps, [pkt.execution_time_ms for pkt in trail],
         color=colors, edgecolor="white", height=0.6)
for i, pkt in enumerate(trail):
    ax1.text(0.5, pkt.step_num, f" {pkt.tool_name}",
             va="center", fontsize=9, fontweight="bold")
ax1.set_ylabel("Step")
ax1.set_title("Governed Pipeline Timeline")
ax1.invert_yaxis()
allowed_patch = mpatches.Patch(color="#2ecc71", label="Allowed")
blocked_patch = mpatches.Patch(color="#e74c3c", label="Blocked")
ax1.legend(handles=[allowed_patch, blocked_patch], loc="lower right")

# Bottom: verification scores
verif_steps = []
verif_rates = []
for pkt in trail:
    if pkt.verification is not None:
        verif_steps.append(pkt.step_num)
        verif_rates.append(pkt.verification["pass_rate"])

if verif_steps:
    ax2.bar(verif_steps, verif_rates, color="#3498db", width=0.5)
    ax2.axhline(y=0.80, color="orange", linestyle="--", label="80% threshold")
    ax2.set_ylabel("Verification\nPass Rate")
    ax2.set_xlabel("Step")
    ax2.set_ylim(0, 1.1)
    ax2.legend(loc="lower right")

plt.tight_layout()
plt.savefig("governance_monograph/notebooks/ch14_timeline.png", dpi=100,
            bbox_inches="tight")
plt.show()
print("Timeline saved to ch14_timeline.png")

Timeline saved to ch14_timeline.png


### Exercise 14.2 — Run Comparison

Run the pipeline a second time with a stricter plausibility threshold and compare the two runs.

In [21]:
# Second run with stricter threshold
RUN_ID_2 = "ch14_strict_run"
tracer.start_run(RUN_ID_2, description="Strict plausibility run")

# Create a stricter gateway
session2 = SessionState()
gateway_strict = GovernedToolGateway(
    session=session2,
    store=store,
    policy=policy,
    theta_plausibility=0.8,  # much stricter
    device=str(DEVICE),
)

current_state = "init"
for step_num, (tool_name, args, context, state_before, state_after) in enumerate(
    pipeline_steps, 1
):
    gw_result = gateway_strict.execute(tool_name, args, context)
    decision = "blocked" if gw_result.blocked else "allowed"
    blocking = []
    if gw_result.blocked and gw_result.policy_decision is not None:
        if not gw_result.policy_decision.allowed:
            blocking = gw_result.policy_decision.blocking_rules
    
    gms_scores = {}
    if gw_result.plausibility_score is not None:
        gms_scores["plausibility"] = gw_result.plausibility_score
    
    next_state = state_after if not gw_result.blocked else current_state
    
    tracer.trace_tool_call(
        run_id=RUN_ID_2,
        step_num=step_num,
        tool_name=tool_name,
        args=args,
        result_summary=gw_result.reason[:100],
        policy_decision=decision,
        blocking_rules=blocking,
        gms_scores=gms_scores,
        gate_results=gw_result.gate_results,
        execution_time_ms=gw_result.execution_time_ms,
        state_before=current_state,
        state_after=next_state,
    )
    current_state = next_state

tracer.end_run(RUN_ID_2)

# Compare runs
trail1 = tracer.get_audit_trail(RUN_ID)
trail2 = tracer.get_audit_trail(RUN_ID_2)

blocked1 = len(tracer.query_blocked_calls(RUN_ID))
blocked2 = len(tracer.query_blocked_calls(RUN_ID_2))

print("=== Run Comparison ===")
print(f"{'Metric':<30s} {'Run 1 (theta=1.5)':<20s} {'Run 2 (theta=0.8)'}")
print("-" * 70)
print(f"{'Total calls':<30s} {len(trail1):<20d} {len(trail2)}")
print(f"{'Blocked':<30s} {blocked1:<20d} {blocked2}")
print(f"{'Block rate':<30s} {blocked1/len(trail1):<20.1%} {blocked2/len(trail2):.1%}")

# Per-step comparison
print(f"\n{'Step':<5s} {'Tool':<22s} {'Run 1':<12s} {'Run 2':<12s} {'Changed'}")
print("-" * 60)
for p1, p2 in zip(trail1, trail2):
    changed = "YES" if p1.policy_decision != p2.policy_decision else ""
    print(f"{p1.step_num:<5d} {p1.tool_name:<22s} {p1.policy_decision:<12s} "
          f"{p2.policy_decision:<12s} {changed}")

=== Run Comparison ===
Metric                         Run 1 (theta=1.5)    Run 2 (theta=0.8)
----------------------------------------------------------------------
Total calls                    7                    7
Blocked                        1                    1
Block rate                     14.3%                14.3%

Step  Tool                   Run 1        Run 2        Changed
------------------------------------------------------------
1     load_dataset           allowed      allowed      
2     preprocess_data        allowed      allowed      
3     train_model            allowed      allowed      
4     predict                blocked      blocked      
5     diagnose_robustness    allowed      allowed      
6     explain_importance     allowed      allowed      
7     data_summary           allowed      allowed      


### Exercise 14.3 — Audit Report

Generate a structured audit report suitable for a model risk manager.

In [22]:
def generate_audit_report(tracer, run_id):
    """Generate a formatted audit report for model risk review."""
    trail = tracer.get_audit_trail(run_id)
    blocked = tracer.query_blocked_calls(run_id)
    verifs = tracer.get_verifications(run_id)
    transitions = tracer.get_transitions(run_id)
    meta = tracer._run_meta.get(run_id, {})
    
    lines = []
    lines.append("=" * 65)
    lines.append("  MODEL RISK AUDIT REPORT")
    lines.append("=" * 65)
    lines.append(f"  Run ID:      {run_id}")
    lines.append(f"  Description: {meta.get('description', 'N/A')}")
    elapsed = meta.get('elapsed_s', 0.0)
    lines.append(f"  Duration:    {elapsed:.2f} s")
    lines.append("")
    
    # Section 1: Policy Compliance
    lines.append("  --- 1. POLICY COMPLIANCE ---")
    lines.append(f"  Total tool calls:  {len(trail)}")
    lines.append(f"  Allowed:           {len(trail) - len(blocked)}")
    lines.append(f"  Blocked:           {len(blocked)}")
    lines.append(f"  Compliance rate:   {(len(trail)-len(blocked))/len(trail):.1%}")
    lines.append("")
    
    # Section 2: Verification Results
    lines.append("  --- 2. VERIFICATION RESULTS ---")
    n_claims = 0
    n_passed = 0
    severities = {}
    for v_rec in verifs:
        for verdict in v_rec.get("verdicts", []):
            n_claims += 1
            if verdict.get("passed"):
                n_passed += 1
            sev = verdict.get("severity", "unknown")
            severities[sev] = severities.get(sev, 0) + 1
    lines.append(f"  Claims verified:   {n_claims}")
    lines.append(f"  Claims passed:     {n_passed}")
    lines.append(f"  Pass rate:         {n_passed/n_claims:.1%}" if n_claims > 0 else "  Pass rate:         N/A")
    lines.append(f"  Severity dist:     {severities}")
    lines.append("")
    
    # Section 3: Exception Log
    lines.append("  --- 3. EXCEPTION LOG ---")
    if blocked:
        for b in blocked:
            rules = ", ".join(b.blocking_rules) if b.blocking_rules else "gate"
            lines.append(f"  [{b.step_num}] {b.tool_name}: {b.result_summary[:60]}")
            lines.append(f"       Blocked by: {rules}")
    else:
        lines.append("  No exceptions.")
    lines.append("")
    
    # Section 4: Decision Rationale
    lines.append("  --- 4. DECISION RATIONALE (per step) ---")
    for pkt in trail:
        plaus = pkt.gms_scores.get("plausibility")
        plaus_str = f"plausibility={plaus:.4f}" if plaus is not None else "no plausibility"
        verif_str = ""
        if pkt.verification is not None:
            v = pkt.verification
            verif_str = f", verified={v['n_passed']}/{v['n_claims']}"
        lines.append(f"  [{pkt.step_num}] {pkt.tool_name}: "
                     f"{pkt.policy_decision} ({plaus_str}{verif_str})")
    lines.append("")
    lines.append("=" * 65)
    
    return "\n".join(lines)

report = generate_audit_report(tracer, RUN_ID)
print(report)

  MODEL RISK AUDIT REPORT
  Run ID:      ch14_demo_run
  Description: Chapter 14 governed pipeline demo
  Duration:    1.51 s

  --- 1. POLICY COMPLIANCE ---
  Total tool calls:  7
  Allowed:           6
  Blocked:           1
  Compliance rate:   85.7%

  --- 2. VERIFICATION RESULTS ---
  Claims verified:   6
  Claims passed:     4
  Pass rate:         66.7%
  Severity dist:     {'error': 2, 'info': 4}

  --- 3. EXCEPTION LOG ---
  [4] predict: Policy violation: Rule block_predict_pending_approval: 'pred
       Blocked by: block_predict_pending_approval, gms_graph:predict->blocks->prediction_approval

  --- 4. DECISION RATIONALE (per step) ---
  [1] load_dataset: allowed (no plausibility, verified=0/1)
  [2] preprocess_data: allowed (no plausibility)
  [3] train_model: allowed (no plausibility, verified=2/3)
  [4] predict: blocked (no plausibility)
  [5] diagnose_robustness: allowed (no plausibility, verified=2/2)
  [6] explain_importance: allowed (no plausibility)
  [7] data_summary:

---
## Lab 14.3 — Audit-Defensible Hardening

The labs so far built a queryable trail. An independent reviewer's next questions are sharper:

1. How do you know nobody edited the trail after the run? **Hash chain.**
2. How do you know the trail captured everything that actually happened? **Cross-reference attestor.**
3. Show me the attack classes you tested against. **Systematic adversarial cohort.**
4. Do summary metrics reproduce byte-identically across runs? **Deterministic replay fingerprint.**
5. Does the bundle authenticate its producer? **HMAC-signed `AuditTrailBundle`.**

The library at `governed_agent/tracing_hardening/` answers these. Five modules: `hash_chain`, `cross_attestor`, `audit_adversarial`, `audit_replay`, `audit_bundle`.

### Lab 14.3 §1 — Build trail, seal, verify

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    make_baseline_trail, seal_audit_trail, verify_audit_chain,
    chain_head_hash,
)

# Build a representative trail (3 allowed + 1 blocked) plus the
# matching runtime-decision log.
packets, runtime_events = make_baseline_trail()
print(f"trail length: {len(packets)} packets")
for p in packets:
    print(f"  [{p.step_num}] {p.tool_name}: {p.policy_decision}")

# Seal: stamps content_hash + prev_hash on every packet.
seal_audit_trail(packets)
print(f"\nchain head: {chain_head_hash(packets)[:24]}...")

# Verify: zero breaks on intact trail.
breaks = verify_audit_chain(packets)
print(f"verify breaks: {len(breaks)}")


### Lab 14.3 §2 — Cross-attestor against runtime decision log

In [ ]:
from knowlytix.harness.governance.tracing_hardening import cross_attest_policy

cross_breaks = cross_attest_policy(runtime_events, packets)
print(f"cross-attestor breaks (canonical trail): {len(cross_breaks)}")

# Now drop the blocked packet from the audit trail (simulates a
# misconfigured tracer); the cross-attestor must catch it.
import copy
tampered = copy.deepcopy(packets[:2] + packets[3:])
breaks_drop = cross_attest_policy(runtime_events, tampered)
print("\nafter dropping blocked packet:")
for b in breaks_drop:
    print(f"  [{b.kind}] step={b.step_num}: {b.detail[:80]}")


### Lab 14.3 §3 — v1 adversarial sweep (9 probes)

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    build_audit_adversarial_cohort, sweep_audit_adversarial,
)

cohort = build_audit_adversarial_cohort()
report = sweep_audit_adversarial(cohort)
print(report.summary())


### Lab 14.3 §4 — Build signed AuditTrailBundle

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    AuditOperatingPoint, build_audit_trail_bundle,
    bind_audit_from_bundle,
    replay_audit_metrics, metrics_fingerprint,
)

# Replay metrics to capture the deterministic fingerprint
metrics = replay_audit_metrics("lab-14-3", packets)
fp = metrics_fingerprint(metrics)
print(f"metrics fingerprint: {fp[:24]}...")

# Build the bundle
op = AuditOperatingPoint(
    min_cohort_safety_rate=1.0,
    required_attestor_coverage="both",
)
bundle = build_audit_trail_bundle(
    run_id="lab-14-3", packets=packets,
    cohort_report=report,
    attestors_run=("chain", "cross"),
    operating_point=op,
)
print(f"bundle id: {bundle.bundle_id}")
print(f"signing method: {bundle.signing_method}")

# Bind: refuses if chain, metrics or SLO drift.
bound = bind_audit_from_bundle(bundle, packets=packets, replay_result=metrics)
print("\nbound: chain head + metrics + SLO all match")


---
## Lab 14.4 — Production Hardening Beyond Audit-Defensible

Five additional gaps closed on top of Lab 14.3 to bring the audit surface to the same standard as Ch 12:

1. v2 cohort: 9 → 60 probes across 10 attack classes (positions/indices/payloads vary)
2. FSM invariants: 29/29 reachable states pass J1–J5 (including the structural property "tail truncation is invisible to the chain alone")
3. Independent alt-validator: recursive functional walk vs iterative — agreement on every chain probe
4. k-of-n multi-signer workflow (reuses Ch 12 `policy_multisign` API)
5. Latency benchmark: seal/verify/replay at n_packets ∈ {10, 100, 1000}

Full integrated runbook: `gms_agent_builder_ext/lab14_audit_trail_runbook.py` (10 sections). Ship gate: `scripts/gate_audit_bundle.py`. Model card: `gms_agent_builder_ext/MODEL_CARD_audit_bundle.md`.

### Lab 14.4 §1 — v2 adversarial cohort (60 probes)

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    build_audit_adversarial_cohort_v2, sweep_audit_adversarial,
)

cohort_v2 = build_audit_adversarial_cohort_v2(n_per_type=6)
print(f"v2 cohort: {len(cohort_v2)} probes")

from collections import Counter
attack_classes = Counter(p.name.split('_v')[0].split('_at_')[0] for p in cohort_v2)
for cls, n in sorted(attack_classes.items()):
    print(f"  {cls}: {n}")


In [28]:
report_v2 = sweep_audit_adversarial(cohort_v2)
print(report_v2.summary())


AuditAdversarialReport(
  n_total=60
  safety_rate=1.000
    chain: 35/35 safe (100.00%)
    cross: 25/25 safe (100.00%)
)


### Lab 14.4 §2 — FSM invariants (29 reachable states)

In [ ]:
from knowlytix.harness.governance.tracing_hardening import verify_audit_invariants

inv = verify_audit_invariants()
print(inv.summary())
assert inv.all_pass, "FSM invariant violation -- ship blocker"


### Lab 14.4 §3 — Independent alt-validator agreement

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    assert_chain_alt_agreement,
)

agreement = assert_chain_alt_agreement(cohort_v2)
print(agreement.summary())
assert agreement.all_agree, "Alt-validator disagreement -- ship blocker"


### Lab 14.4 §4 — k-of-n multi-signer audit bundle

In [ ]:
from knowlytix.harness.governance import (
    attach_signatures, sign_for, verify_quorum,
)

# Three deployment attestors; require 2 of 3 for ship.
keys = {
    "mrm_lead":           "deployment-mrm-lead-key-32-bytes!!",
    "compliance_officer": "deployment-compliance-key-32-bytes!",
    "platform_owner":     "deployment-platform-key-32-bytes!!",
}
sigs = [sign_for(bundle, sid, k) for sid, k in keys.items()]
attach_signatures(bundle, sigs)
quorum = verify_quorum(bundle, keys, k=2)
print(quorum.summary())


### Lab 14.4 §5 — Latency benchmark

In [ ]:
from knowlytix.harness.governance.tracing_hardening import (
    bench_seal_audit_trail, bench_verify_audit_chain,
    bench_replay_audit_metrics,
)

print("seal_audit_trail:")
for n in [10, 100, 1000]:
    print(f"  n={n:5d}: {bench_seal_audit_trail(n_packets=n, n_calls=300, warmup=30).summary()}")

print("\nverify_audit_chain:")
for n in [10, 100, 1000]:
    print(f"  n={n:5d}: {bench_verify_audit_chain(n_packets=n, n_calls=300, warmup=30).summary()}")

print("\nreplay_audit_metrics:")
for n in [10, 100, 1000]:
    print(f"  n={n:5d}: {bench_replay_audit_metrics(n_packets=n, n_calls=300, warmup=30).summary()}")
